# Retrieval Using Ensemble (BM25 + SBERT) and Generation using LLM
The purpose of this notebook is to make the **Ensemble** retrieve the appropriate statutes and precedents for the **LLM**. The **LLM** will use those documents as context to generate a simple explaination for the user.

## Install and Import Libraries

In [2]:
!pip install gradio rank_bm25 \
sentence-transformers openai -q

Add the [utilities](https://www.kaggle.com/datasets/mutahir314/utilities) dataset as input to the Kaggle Notebook.

In [3]:
import time
import os
import sys

from huggingface_hub import login
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from openai import OpenAI
import numpy as np
import torch
import nltk
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from sentence_transformers import util
from sentence_transformers import SentenceTransformer
import pickle
import gradio as gr

DATASETS_DIR = f"/kaggle/input/datasets/mutahir314"
DATASET_NAME = "utilities"

if DATASETS_DIR not in sys.path:
    sys.path.append(DATASETS_DIR)

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Add the utilities dataset as input
from utilities import preprocessing
from utilities import loading

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

/kaggle/input/datasets/mutahir314/utilities/preprocessing.py
/kaggle/input/datasets/mutahir314/utilities/statute_embeddings.pt
/kaggle/input/datasets/mutahir314/utilities/evaluation.py
/kaggle/input/datasets/mutahir314/utilities/query_embeddings.pt
/kaggle/input/datasets/mutahir314/utilities/loading.py
/kaggle/input/datasets/mutahir314/utilities/bm25_indexes.pkl
/kaggle/input/datasets/mutahir314/utilities/precedent_embeddings.pt


True

## Load Dataset

Get your HuggingFace token from [HuggingFace](https://huggingface.co/settings/tokens) and then go to _Add-ons->Secrets->Add Secret_ in the Kaggle Notebook. Label the key `HF_TOKEN` and assign it the value of the token you got.

In [4]:
# Access the token saved in Kaggle Secrets
user_secrets = UserSecretsClient()
huggingface_token = user_secrets.get_secret("HF_TOKEN")
data = loading.load_ilpcsr(huggingface_token, verbose=True)

README.md: 0.00B [00:00, ?B/s]

train_queries.parquet:   0%|          | 0.00/48.4M [00:00<?, ?B/s]

dev_queries.parquet:   0%|          | 0.00/6.54M [00:00<?, ?B/s]

test_queries.parquet:   0%|          | 0.00/6.45M [00:00<?, ?B/s]

Generating train_queries split:   0%|          | 0/5017 [00:00<?, ? examples/s]

...

statute_candidates.parquet:   0%|          | 0.00/1.17M [00:00<?, ?B/s]

Generating statute_candidates split:   0%|          | 0/936 [00:00<?, ? examples/s]

precedent_candidates.parquet:   0%|          | 0.00/72.4M [00:00<?, ?B/s]

Generating precedent_candidates split:   0%|          | 0/3183 [00:00<?, ? examples/s]

Statutes Shape: 936
Precedent Shape: 3183
Query Shape: 627


## Prepare Text

In [5]:
statute_texts_clean   = [preprocessing.prepare_text(t)
    for t in data["statute_texts"]]
precedent_texts_clean = [preprocessing.prepare_text(t)
    for t in data["precedent_texts"]]

## Upload Embeddings

The embeddings were created using [SBERT](https://sbert.net/) in another jupyter notebook. It is expensive to create embeddings so they were stored in a kaggle dataset so they can be easily accessed.

In [6]:
statute_embeddings   = torch.load(
    f'{DATASETS_DIR}/{DATASET_NAME}/statute_embeddings.pt',
    map_location='cpu')
precedent_embeddings = torch.load(
    f'{DATASETS_DIR}/{DATASET_NAME}/precedent_embeddings.pt',
    map_location='cpu')

print(f"Statute embeddings  : "
      f"{statute_embeddings.shape}")
print(f"Precedent embeddings: "
      f"{precedent_embeddings.shape}")

Statute embeddings  : torch.Size([936, 384])
Precedent embeddings: torch.Size([3183, 384])


## Build BM25 Indexes Once

The BM25 indexes were also computed beforehand and put into the dataset mentioned before.

In [7]:
bm25_indexes_path = f"{DATASETS_DIR}/{DATASET_NAME}/bm25_indexes.pkl"
if os.path.exists(bm25_indexes_path):
    print("bm25_indexes.pkl already exists. Loading it!")
    with open(bm25_indexes_path, "rb") as f:
        bm25_indexes = pickle.load(f)
else:
    bm25_stat = BM25Okapi(
    [preprocessing.tokenize_ngram(t, 3) for t in data["statute_texts"]],
    k1=1.6, b=0.75)

    bm25_prec = BM25Okapi(
        [preprocessing.tokenize_ngram(t, 5) for t in data["precedent_texts"]],
        k1=1.6, b=0.75)

    bm25_indexes = {
            "bm25_statute": bm25_stat,
            "bm25_precedent": bm25_prec
    }

    # This file will be created in /kaggle/working/ directory
    with open("bm25_indexes.pkl", "wb") as f:
        pickle.dump(bm25_indexes, f)

bm25_indexes.pkl already exists. Loading it!


## Load SBERT Model

In [8]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Setup Groq

Get your own token from [Groq](https://console.groq.com/home) and make a secret key with the label `GROQ_API_KEY` with the token as its value.

In [9]:
user_secrets = UserSecretsClient()
grok_api_key = user_secrets.get_secret("GROQ_API_KEY")

client = OpenAI(
    api_key=grok_api_key,
    base_url="https://api.groq.com/openai/v1"
)

## Ensemble Retrieval Function

In [10]:
def ensemble_retrieve(query, k=3,
                      alpha_stat=0.0,
                      alpha_prec=0.5):
    """
    Ensemble retrieval for RAG
    alpha=0.0 → pure BM25 (best for LSR)
    alpha=0.5 → equal mix (best for PCR)
    """
    # Encode query with SBERT
    q_clean = preprocessing.prepare_text(query)
    q_emb = sbert_model.encode(
        [q_clean],
        convert_to_tensor=True)

    # ── Statute Retrieval ──────────────────
    # BM25 scores
    qt3 = preprocessing.tokenize_ngram(query, 3)
    bm25_s = bm25_indexes["bm25_statute"].get_scores(qt3)
    bm25_s_norm = (bm25_s - bm25_s.mean()) / \
                  (bm25_s.std() + 1e-8)

    # SBERT scores
    sbert_s = util.cos_sim(
        q_emb, statute_embeddings)[0]\
        .cpu().numpy()
    sbert_s_norm = (sbert_s - sbert_s.mean()) / \
                   (sbert_s.std() + 1e-8)

    # Ensemble
    ens_s = alpha_stat * sbert_s_norm + \
            (1 - alpha_stat) * bm25_s_norm
    top_stat = np.argsort(ens_s)[::-1][:k]

    # ── Precedent Retrieval ────────────────
    # BM25 scores
    qt5 = preprocessing.tokenize_ngram(query, 5)
    bm25_p = bm25_indexes["bm25_precedent"].get_scores(qt5)
    bm25_p_norm = (bm25_p - bm25_p.mean()) / \
                  (bm25_p.std() + 1e-8)

    # SBERT scores
    sbert_p = util.cos_sim(
        q_emb, precedent_embeddings)[0]\
        .cpu().numpy()
    sbert_p_norm = (sbert_p - sbert_p.mean()) / \
                   (sbert_p.std() + 1e-8)

    # Ensemble
    ens_p = alpha_prec * sbert_p_norm + \
            (1 - alpha_prec) * bm25_p_norm
    top_prec = np.argsort(ens_p)[::-1][:k]

    return (
        [statute_texts_clean[i] for i in top_stat],
        [data["statute_ids"][i] for i in top_stat],
        [precedent_texts_clean[i] for i in top_prec],
        [data["precedent_ids"][i] for i in top_prec]
    )

## Retrieval Augmented Generation (RAG)

In [11]:
def rag_with_ensemble(query):
    """Full RAG pipeline with ensemble retrieval"""

    # Retrieve
    (top_statutes, stat_ids,
     top_precedents, prec_ids) = \
        ensemble_retrieve(query, k=3)

    # Build context
    context  = "RELEVANT STATUTES:\n"
    for i, (sid, stat) in enumerate(
            zip(stat_ids, top_statutes)):
        context += f"\nStatute {i+1} "
        context += f"(ID: {sid}):\n"
        context += stat[:500] + "\n"

    context += "\n\nRELEVANT PRECEDENTS:\n"
    for i, (pid, prec) in enumerate(
            zip(prec_ids, top_precedents)):
        context += f"\nPrecedent {i+1} "
        context += f"(ID: {pid}):\n"
        context += prec[:500] + "\n"

    prompt = f"""You are a helpful legal assistant.

    Based ONLY on the provided statutes and precedents,
    explain in simple plain English:
    1. Which specific laws apply
    2. What those laws say simply
    3. What similar cases decided
    4. Practical steps to take next
    
    Keep under 300 words. Simple language.
    End with: DISCLAIMER: This is not legal advice.
    Please consult a qualified lawyer.
    
    SITUATION: {query}
    
    {context}"""

    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system",
                 "content": "You are a helpful "
                 "legal assistant."},
                {"role": "user",
                 "content": prompt}
            ],
            max_tokens=500,
            temperature=0.3
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"Error: {str(e)}"

## Making a basic UI with [Gradio](https://www.gradio.app/)

In [12]:
def legal_query(query, show_retrieved):
    """Main function for Gradio interface"""

    if not query.strip():
        return "Please enter your legal situation.", ""

    # Retrieve documents
    (top_statutes, stat_ids,
     top_precedents, prec_ids) = \
        ensemble_retrieve(query, k=3)

    # Generate explanation
    explanation = rag_with_ensemble(query)

    # Build retrieved docs display
    if show_retrieved:
        retrieved = "### Retrieved Statutes\n\n"
        for i, (sid, stat) in enumerate(
                zip(stat_ids, top_statutes)):
            retrieved += f"**Statute {i+1}** "
            retrieved += f"(ID: {sid})\n"
            retrieved += stat[:300] + "...\n\n"

        retrieved += "---\n\n"
        retrieved += "### Retrieved Precedents\n\n"
        for i, (pid, prec) in enumerate(
                zip(prec_ids, top_precedents)):
            retrieved += f"**Precedent {i+1}** "
            retrieved += f"(ID: {pid})\n"
            retrieved += prec[:300] + "...\n\n"
    else:
        retrieved = f"Retrieved {len(stat_ids)} " \
                    f"statutes and " \
                    f"{len(prec_ids)} precedents.\n\n" \
                    f"Enable 'Show Retrieved Docs' " \
                    f"to see them."

    return explanation, retrieved

# ── Build Gradio Interface ─────────────────
with gr.Blocks(
    title="LegalIR-PCSR",
    theme=gr.themes.Default()
) as demo:

    gr.Markdown("""
    # LegalIR-PCSR
    ## Automated Legal Information Retrieval System
    *Air University Islamabad | Information Retrieval AI323*

    Describe your legal situation in plain English.
    The system retrieves relevant laws and court cases,
    then explains your rights in simple language.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            query_input = gr.Textbox(
                label="Describe Your Legal Situation",
                placeholder="Example: My employer has not "
                            "paid my salary for 3 months. "
                            "What law protects me?",
                lines=5
            )

            show_docs = gr.Checkbox(
                label="Show Retrieved Documents",
                value=False
            )

            submit_btn = gr.Button(
                "Find Legal Information",
                variant="primary"
            )

            gr.Examples(
                examples=[
                    ["My employer has not paid my salary "
                     "for 3 months. I have asked multiple "
                     "times but he refuses. What law "
                     "protects me?"],
                    ["My landlord illegally cut my "
                     "electricity without any notice or "
                     "court order. I pay rent on time. "
                     "What are my rights?"],
                    ["I bought a mobile phone online and "
                     "received a fake item. The seller "
                     "refuses to refund. What can I do?"]
                ],
                inputs=query_input,
                label="Example Queries"
            )

        with gr.Column(scale=1):
            explanation_output = gr.Textbox(
                label="Legal Explanation",
                lines=15,
                interactive=False
            )

            retrieved_output = gr.Textbox(
                label="Retrieved Documents",
                lines=8,
                interactive=False
            )

    submit_btn.click(
        fn=legal_query,
        inputs=[query_input, show_docs],
        outputs=[explanation_output, retrieved_output]
    )

    gr.Markdown("""
    ---
    **Important Disclaimer**
    This system is for research and educational
    purposes only. All outputs are informational
    and do NOT constitute legal advice.
    Always consult a qualified lawyer.

    *Dataset: IL-PCSR (Paul et al., EMNLP 2025)*
    """)

# ── Launch ─────────────────────────────────
print("Launching LegalIR-PCSR...")
demo.launch(
    share=True,
    debug=False
)

/tmp/ipykernel_58/3622806320.py:41: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(


Launching LegalIR-PCSR...
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c94c47aef70fedf4a6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
